# 10번. Train,Test셋 나누기

In [1]:
import pandas as pd

df = pd.read_parquet("7번. 산업별 데이터/M19_도매_소매업.parquet")  # <- 이부분을 영경님이 데이터 제작 완료하신다면, '8,9번. +산업별 평균 활용데이터/M19_도매_소매업.parquet' 로 바꿔서 실행시켜주세요

# ── Case 1. 연도 컬럼이 별도로 있는 경우 ──────────────────
# 예) df 컬럼: ['회사코드', '회계년도', '부채비율', ...]

train = df[df["회계년도"].between(2012, 2022)].copy()
test  = df[df["회계년도"].between(2023, 2024)].copy()



# ── Case 2. 연도가 인덱스인 경우 ──────────────────────────
# 예) df.index = [2012, 2013, ..., 2024]

train = df.loc[2012:2022]
test  = df.loc[2023:2024]


# ── 확인 ──────────────────────────────────────────────────
print("Train shape:", train.shape)  # 2012~2022: 11개 연도
print("Test  shape:", test.shape)   # 2023~2024:  2개 연도

Train shape: (11, 74)
Test  shape: (2, 74)


행(row) = 연도 1개

→ train: 2012~2022 = 11행, test: 2023~2024 = 2행 

# 11번. 이상치 탐지 및 처리


이상치 탐지는 IQR(Interquartile Range) 방식을 사용하였다. 구체적으로, Train 데이터(2012~2022)를 기준으로 각 변수의 1사분위수(Q1)와 3사분위수(Q3)를 계산한 후 IQR(Q3-Q1)을 산출하였다. 이후 Q1 - 1.5 × IQR 미만, Q3 + 1.5 × IQR 초과하는 값을 이상치로 판단하였다.

탐지된 이상치는 윈저라이징(Winsorizing) 방식으로 처리하였다. 윈저라이징은 이상치를 제거하는 대신 각 경계값으로 대체하는 방법으로, 데이터 손실 없이 극단값의 영향을 완화할 수 있다. 또한 데이터 누수(Data Leakage) 방지를 위해 Train 데이터에서 산출한 경계값을 Test 데이터(2023~2024)에도 동일하게 적용하였다.

In [2]:
# ── 이상치 탐지 대상 컬럼 (라벨·메타 제외) ───────────────
exclude_cols = ["회사코드", "회계년도", "부실라벨_ICR3년", "빅4감사", 
                '사업자등록번호', '회사명', '통계청 한국표준산업분류 11차(중분류)',
                '통계청 한국표준산업분류 11차(대분류)', 'M코드']
feature_cols = [c for c in train.columns 
                if c not in exclude_cols 
                and train[c].dtype in ["float64", "float32", "int64", "int32"]]

# ── 1. Train 기준으로 IQR 경계값 계산 ───────────────────
lower_bound = {}
upper_bound = {}

for col in feature_cols:
    Q1  = train[col].quantile(0.25)
    Q3  = train[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound[col] = Q1 - 1.5 * IQR
    upper_bound[col] = Q3 + 1.5 * IQR

# ── 2. 이상치 탐지 리포트 (클리핑 전) ───────────────────
print("=" * 75)
print("[ Train 이상치 탐지 결과 - IQR 방식 ]")
print("=" * 75)
print(f"{'컬럼':<25} {'이상치 수':>8} {'비율(%)':>8} {'하한(Q1-1.5IQR)':>16} {'상한(Q3+1.5IQR)':>16}")
print("-" * 75)

outlier_report = {}
for col in feature_cols:
    n_lower = (train[col] < lower_bound[col]).sum()
    n_upper = (train[col] > upper_bound[col]).sum()
    n_total = n_lower + n_upper
    ratio   = n_total / len(train) * 100
    outlier_report[col] = n_total
    if n_total > 0:
        print(f"{col:<25} {n_total:>8} {ratio:>7.2f}%"
              f" {lower_bound[col]:>16.4f} {upper_bound[col]:>16.4f}")

print(f"\n→ 이상치 존재 컬럼 수: {sum(v > 0 for v in outlier_report.values())}개 / 전체 {len(feature_cols)}개")

# ── 3. 윈저라이징 적용 ───────────────────────────────────
for col in feature_cols:
    train.loc[:, col] = train[col].clip(lower=lower_bound[col],
                                        upper=upper_bound[col])
    test.loc[:, col]  = test[col].clip(lower=lower_bound[col],
                                       upper=upper_bound[col])

print("\n✅ 윈저라이징 완료")

[ Train 이상치 탐지 결과 - IQR 방식 ]
컬럼                           이상치 수    비율(%)    하한(Q1-1.5IQR)    상한(Q3+1.5IQR)
---------------------------------------------------------------------------
종업원                              2   18.18%         158.0000         158.0000
총부채비율                            1    9.09%          45.9193          96.2216
차입금의존도                           1    9.09%          24.7634          70.3552
자기자본비율                           1    9.09%           3.7827          54.0794
유보율                              2   18.18%      -34934.5833      123982.0833
자본잠식률                            2   18.18%       -1294.4333         316.2333
유형자산비율                           1    9.09%          37.6145          63.2783
ROA                              2   18.18%          -3.9426          11.6694
ROE                              2   18.18%          -6.1294          35.5924
ROIC                             2   18.18%          -1.9570          16.2956
총자본영업이익률                         2   

# 12번. 스케일링

재무비율 데이터는 변수 간 단위가 어느 정도 통일되어 있으나, 비율 간에도 값의 범위가 크게 다르고(예: 부채비율 수백 % vs 총자산회전율 1~2회) 극단값이 빈번하게 나타나는 특성이 있다. 이에 이상치에 강건한 Robust Scaler를 적용하였다. Robust Scaler는 평균과 표준편차 대신 중앙값(Median)과 IQR을 기준으로 데이터를 변환하기 때문에 극단값의 영향을 최소화할 수 있으며, 앞서 IQR 기반의 윈저라이징을 적용한 것과 동일한 기준을 사용한다는 점에서 일관성이 있다.

스케일링은 데이터 누수(Data Leakage) 방지를 위해 Train 데이터(2012~2022)에 대해서만 fit_transform을 수행하여 중앙값과 IQR을 학습하였으며, Test 데이터(2023~2024)에는 Train 기준으로 학습된 값을 그대로 적용하는 transform만 수행하였다.

In [3]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()

# Train : fit + transform (중앙값/IQR 학습 후 변환)
train.loc[:, feature_cols] = scaler.fit_transform(train[feature_cols])

# Test : transform만 (Train 기준 중앙값/IQR 그대로 사용)
test.loc[:, feature_cols]  = scaler.transform(test[feature_cols])

이해 차원에서 적어둠
- fit : 데이터의 통계값(중앙값, IQR)을 학습하는 것
- transform : 학습한 통계값으로 데이터를 변환하는 것

# 저장

In [4]:
train.to_parquet('10,11,12번 [Train] .parquet')
train.to_parquet('10,11,12번 [Test].parquet')